In [ ]:
import sys
import torch
import os
from pathlib import Path

print("=== Ambiente ===")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n=== Inputs ===")
for item in sorted(os.listdir('/kaggle/input/')):
    print(f"  /kaggle/input/{item}")

In [ ]:
!pip install facenet-pytorch --no-deps -q
!pip install timm -q
print("Dependências instaladas")

In [ ]:
import sys
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
from facenet_pytorch import MTCNN
from PIL import Image
import matplotlib.pyplot as plt

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")       
print(f"Pandas: {pd.__version__}")        
print(f"OpenCV: {cv2.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(image_size=224, device=device)
print(f"\n MTCNN inicializado em {device}")

In [ ]:
POSSIBLE_PATHS = [
    '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
]
VIDEOS_PATH = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        VIDEOS_PATH = Path(p)
        break

assert VIDEOS_PATH is not None, "Nenhum caminho do DFDC encontrado!"
print(f"DFDC em: {VIDEOS_PATH}")

WORK_DIR = Path('/kaggle/working')
FACES_DIR = WORK_DIR / 'faces'
FACES_DIR.mkdir(exist_ok=True, parents=True)

FRAMES_PER_VIDEO = 10       
IMG_SIZE = 224               
BATCH_SIZE = 32
NUM_EPOCHS = 8
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

print(f"   Configurações definidas")
print(f"   Frames/vídeo: {FRAMES_PER_VIDEO}")
print(f"   Tamanho imagem: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Épocas: {NUM_EPOCHS}")

In [ ]:
with open(VIDEOS_PATH / 'metadata.json', 'r') as f:
    metadata = json.load(f)

df = pd.DataFrame([
    {'filename': k, 'label': v['label'], 'original': v.get('original')}
    for k, v in metadata.items()
])

print(f"Total vídeos: {len(df)}")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nREAL únicos (identities): {df[df['label']=='REAL']['filename'].nunique()}")
print(f"FAKEs com original definido: {df[df['label']=='FAKE']['original'].notna().sum()}")

fakes_per_real = df[df['label']=='FAKE'].groupby('original').size()
print(f"\nFakes por cada REAL:")
print(f"  Média: {fakes_per_real.mean():.1f}")
print(f"  Min: {fakes_per_real.min()}, Max: {fakes_per_real.max()}")

In [ ]:
import random
import numpy as np
import torch

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"SEED: {SEED}")

In [ ]:
POSSIBLE_PATHS = [
    '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
]
VIDEOS_PATH = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        VIDEOS_PATH = Path(p)
        break

assert VIDEOS_PATH is not None, "Nenhum caminho do DFDC encontrado!"
print(f" DFDC em: {VIDEOS_PATH}")

# Pasta de trabalho (para guardar faces extraídas, modelos, etc)
WORK_DIR = Path('/kaggle/working')
FACES_DIR = WORK_DIR / 'faces'
FACES_DIR.mkdir(exist_ok=True, parents=True)

# === HIPERPARÂMETROS ===
FRAMES_PER_VIDEO = 10        
IMG_SIZE = 224               
BATCH_SIZE = 32
NUM_EPOCHS = 8
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

print(f"   Configurações definidas")
print(f"   Frames/vídeo: {FRAMES_PER_VIDEO}")
print(f"   Tamanho imagem: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Épocas: {NUM_EPOCHS}")with open(VIDEOS_PATH / 'metadata.json', 'r') as f:
    metadata = json.load(f)

# DataFrame para facilitar análise
df = pd.DataFrame([
    {'filename': k, 'label': v['label'], 'original': v.get('original')}
    for k, v in metadata.items()
])

print(f"Total vídeos: {len(df)}")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nREAL únicos (identities): {df[df['label']=='REAL']['filename'].nunique()}")
print(f"FAKEs com original definido: {df[df['label']=='FAKE']['original'].notna().sum()}")

# Quantos fakes por cada real?
fakes_per_real = df[df['label']=='FAKE'].groupby('original').size()
print(f"\nFakes por cada REAL:")
print(f"  Média: {fakes_per_real.mean():.1f}")
print(f"  Min: {fakes_per_real.min()}, Max: {fakes_per_real.max()}")

In [ ]:
with open(VIDEOS_PATH / 'metadata.json', 'r') as f:
    metadata = json.load(f)

# DataFrame para facilitar análise
df = pd.DataFrame([
    {'filename': k, 'label': v['label'], 'original': v.get('original')}
    for k, v in metadata.items()
])

print(f"Total vídeos: {len(df)}")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nREAL únicos (identities): {df[df['label']=='REAL']['filename'].nunique()}")
print(f"FAKEs com original definido: {df[df['label']=='FAKE']['original'].notna().sum()}")

# Quantos fakes por cada real?
fakes_per_real = df[df['label']=='FAKE'].groupby('original').size()
print(f"\nFakes por cada REAL:")
print(f"  Média: {fakes_per_real.mean():.1f}")
print(f"  Min: {fakes_per_real.min()}, Max: {fakes_per_real.max()}")

In [ ]:
real_identities = df[df['label']=='REAL']['filename'].tolist()
random.seed(SEED)
random.shuffle(real_identities)

n = len(real_identities)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_ids = set(real_identities[:n_train])
val_ids = set(real_identities[n_train:n_train+n_val])
test_ids = set(real_identities[n_train+n_val:])

print(f"Identities: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")

def assign_split(row):
    if row['label'] == 'REAL':
        identity = row['filename']
    else:
        identity = row['original']
    if identity in train_ids:
        return 'train'
    elif identity in val_ids:
        return 'val'
    elif identity in test_ids:
        return 'test'
    else:
        return 'unknown'

df['split'] = df.apply(assign_split, axis=1)

# Verificar distribuição
print("\nVídeos por split e label:")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))